# ДЗ 2.2 — Стратегии JOIN

Собрал аудиторию кампаний по домену email. Для каждой отправки выбираю версию профиля, действовавшую на send_at. SCD2 построил скриптом из ДЗ 2.1 на 5 млн событий.

## Замеры

Сравнил auto, MERGE, broadcast и SHUFFLE_HASH. У auto оставил стандартные настройки Spark. Для остальных отключил AQE и auto broadcast, задал 16 shuffle-партиций.

После прогрева сделал по пять запусков, чередуя порядок. Измерял полную запись в S3 Parquet, без кеша DataFrame и coalesce(1). Стенд: local[4], память JVM 2 ГиБ. Проверки выполнял отдельно от замеров.

Код — task1.py, сбор метрик — common.py.

In [1]:
from task1 import main
main()

task1:auto:-1 3.778s


task1:merge:-1 4.128s


task1:broadcast:-1 2.287s


task1:shuffle_hash:-1 3.528s


task1:auto:0 2.220s


task1:merge:0 3.121s


... (14 outputs left out)

task1:auto:4 2.001s


task1:merge:4 2.638s


task1:broadcast:4 1.882s


task1:shuffle_hash:4 2.696s


In [2]:
from view_results import show_checks, show_measurements, show_plans
show_checks(1)

Строк в аудитории: 500,233
Все четыре варианта совпали. Повторов campaign_id + user_id нет.


In [3]:
show_measurements(1)
show_plans(1)

auto: 5 замеров, медиана 2.08 с (1.93–2.22), файлов 4.
  Shuffle read/write: 0.0/0.0 МиБ; spill memory/disk: 0.0/0.0 МиБ.
broadcast: 5 замеров, медиана 2.02 с (1.83–2.15), файлов 4.
  Shuffle read/write: 0.0/0.0 МиБ; spill memory/disk: 0.0/0.0 МиБ.
merge: 5 замеров, медиана 2.81 с (2.63–3.29), файлов 5.
  Shuffle read/write: 48.3/48.3 МиБ; spill memory/disk: 0.0/0.0 МиБ.
shuffle_hash: 5 замеров, медиана 2.75 с (2.58–4.11), файлов 5.
  Shuffle read/write: 48.3/48.3 МиБ; spill memory/disk: 0.0/0.0 МиБ.
Все замеры: results/task1/measurements.csv
auto: BroadcastHashJoin; AQE = True
broadcast: BroadcastHashJoin; AQE = False
merge: SortMergeJoin; AQE = False
shuffle_hash: ShuffledHashJoin; AQE = False
Полные планы каждого замера: results/task1/plans/


## Вывод

В результате 500 233 строки. Четыре варианта полностью совпали, повторов `(campaign_id, user_id)` нет.

После прогрева сделал по пять замеров:

- Auto: медиана 2.08 с, разброс 1.93–2.22 с.
- Broadcast: медиана 2.02 с, разброс 1.83–2.15 с.
- MERGE: медиана 2.81 с, разброс 2.63–3.29 с.
- SHUFFLE_HASH: медиана 2.75 с, разброс 2.58–4.11 с.

Spark сам выбрал BroadcastHashJoin. Явный broadcast дал тот же оператор, а hints MERGE и SHUFFLE_HASH — SortMergeJoin и ShuffledHashJoin. Конечный план auto проверил через History Server.

Auto и явный broadcast оказались близки: диапазоны времени пересекаются. Здесь оставил бы auto — справочник из 25 кампаний Spark и так передаёт через broadcast.

У MERGE и SHUFFLE_HASH shuffle read и write составили примерно по 48.3 МиБ. Spill не было. Доменов всего пять, поэтому в этих вариантах из 16 партиций работают только пять: увеличение числа партиций само по себе ключи не разделит.

Даже без shuffle остаются чтение SCD2, проверка интервалов и запись результата в S3. Поэтому время не уменьшается пропорционально shuffle. При росте справочника или изменении распределения доменов выбор стратегии нужно проверить заново.

Все замеры, число файлов и метрики сохранил в `results/task1/measurements.csv`, планы — в `results/task1/plans/`.